# LeRobot Dataset Inspector
Loads the exported dataset through the LeRobot v3.0 framework, then visualises episodes.

**Install once:**
```bash
pip install lerobot
```

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

DATASET_ROOT = pathlib.Path("./lerobot_dataset")
REPO_ID      = "local/fr5_teleop"   # arbitrary string; only used in error messages for local datasets

## Load dataset

In [ ]:
dataset = LeRobotDataset(repo_id=REPO_ID, root=DATASET_ROOT)
print(dataset)

## Dataset summary

In [ ]:
print(f"FPS            : {dataset.fps}")
print(f"Episodes       : {dataset.num_episodes}")
print(f"Total frames   : {dataset.num_frames}")
print(f"Features       : {list(dataset.features.keys())}")
print()
# dataset.meta.tasks is a DataFrame: index=task_text (name='task'), column='task_index'
print("Tasks:")
for task_text, row in dataset.meta.tasks.iterrows():
    print(f"  [{int(row['task_index'])}] {task_text or '(no instruction)'}")

## Inspect one episode

In [ ]:
EPISODE_IDX = 0   # change to inspect a different episode

# episode_data_index is built from dataset.meta.episodes
# Each episode row has dataset_from_index and dataset_to_index (global frame range)
ep_meta   = dataset.meta.episodes[EPISODE_IDX]
from_idx  = int(ep_meta["dataset_from_index"])
to_idx    = int(ep_meta["dataset_to_index"])
n_frames  = to_idx - from_idx
print(f"Episode {EPISODE_IDX}: {n_frames} frames  "
      f"(global frames {from_idx}–{to_idx-1})")

In [ ]:
# Pull all frames for this episode
obs_state  = np.stack([dataset[from_idx + i]["observation.state"].numpy() for i in range(n_frames)])
action     = np.stack([dataset[from_idx + i]["action"].numpy()            for i in range(n_frames)])
timestamps = np.array([dataset[from_idx + i]["timestamp"].item()          for i in range(n_frames)])

print(f"obs_state shape : {obs_state.shape}")   # (N, 6)
print(f"action shape    : {action.shape}")       # (N, 7)
print(f"time range      : 0 → {timestamps[-1]:.2f}s")

## Joint positions: commanded vs actual

In [ ]:
cmd    = action[:, :6]
actual = obs_state

fig, axes = plt.subplots(3, 2, figsize=(13, 9), sharex=True)
for j, ax in enumerate(axes.flatten()):
    ax.plot(timestamps, cmd[:, j],    label="cmd",    lw=1.2)
    ax.plot(timestamps, actual[:, j], label="actual", lw=1.2, linestyle="--")
    ax.set_title(f"J{j+1}")
    ax.set_ylabel("deg")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes[-1, 0].set_xlabel("time (s)")
axes[-1, 1].set_xlabel("time (s)")
fig.suptitle(f"Episode {EPISODE_IDX} — Joint positions", fontsize=12)
plt.tight_layout()
plt.show()

## Gripper

In [ ]:
gripper = action[:, 6]

plt.figure(figsize=(10, 3))
plt.plot(timestamps, gripper, lw=1.5)
plt.axhline(0.65, color="green",  ls="--", lw=0.9, label="open threshold")
plt.axhline(0.35, color="orange", ls="--", lw=0.9, label="close threshold")
plt.ylim(-0.05, 1.05)
plt.xlabel("time (s)"); plt.ylabel("gripper_norm")
plt.title(f"Episode {EPISODE_IDX} — Gripper")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## EEF pose

In [ ]:
eef    = np.stack([dataset[from_idx + i]["observation.eef_pose"].numpy() for i in range(n_frames)])
labels = ["x_mm", "y_mm", "z_mm", "rx_deg", "ry_deg", "rz_deg"]

fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for i, (ax, lbl) in enumerate(zip(axes.flatten(), labels)):
    ax.plot(timestamps, eef[:, i], lw=1.2)
    ax.set_title(lbl); ax.grid(True, alpha=0.3)
    if i >= 3: ax.set_xlabel("time (s)")
fig.suptitle(f"Episode {EPISODE_IDX} — EEF pose", fontsize=12)
plt.tight_layout(); plt.show()

## 3-D EEF path

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax  = fig.add_subplot(111, projection="3d")
sc  = ax.scatter(eef[:, 0], eef[:, 1], eef[:, 2], c=timestamps, cmap="viridis", s=4)
plt.colorbar(sc, ax=ax, label="time (s)")
ax.set_xlabel("X (mm)"); ax.set_ylabel("Y (mm)"); ax.set_zlabel("Z (mm)")
ax.set_title(f"Episode {EPISODE_IDX} — EEF 3-D path")
plt.tight_layout(); plt.show()

## Show wrist camera frame

In [ ]:
frame_i = 0   # index within episode

sample = dataset[from_idx + frame_i]
vid_key = "observation.images.wrist_cam"

if vid_key in sample:
    img = sample[vid_key]          # (C, H, W) float tensor [0, 1]
    img_np = img.numpy()
    if img_np.ndim == 3 and img_np.shape[0] in (1, 3):
        img_np = img_np.transpose(1, 2, 0)   # CHW → HWC
    if img_np.dtype != np.uint8:
        img_np = (img_np * 255).clip(0, 255).astype(np.uint8)
    plt.figure(figsize=(6, 4))
    plt.imshow(img_np)
    plt.axis("off")
    plt.title(f"Episode {EPISODE_IDX}, frame {frame_i}")
    plt.tight_layout(); plt.show()
else:
    print("No camera frames in this episode — camera was not attached during recording.")

## Dataset-wide statistics

In [ ]:
# dataset.meta.tasks: DataFrame with task text as index (name='task'), column 'task_index'
task_idx_to_text = {int(row["task_index"]): txt for txt, row in dataset.meta.tasks.iterrows()}

durations, frame_counts, task_labels = [], [], []

for ep_idx in range(dataset.num_episodes):
    ep     = dataset.meta.episodes[ep_idx]
    f_from = int(ep["dataset_from_index"])
    f_to   = int(ep["dataset_to_index"])
    nf     = f_to - f_from
    ts_end = dataset[f_to - 1]["timestamp"].item()
    durations.append(float(ts_end))
    frame_counts.append(nf)
    tidx = int(dataset[f_from]["task_index"].item())
    task_labels.append(task_idx_to_text.get(tidx, str(tidx)))

task_counts = {lbl: task_labels.count(lbl) for lbl in set(task_labels)}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(durations,    bins=max(1, min(20, len(durations))), color="steelblue", edgecolor="white")
axes[0].set_xlabel("duration (s)"); axes[0].set_title("Episode durations")
axes[1].hist(frame_counts, bins=max(1, min(20, len(frame_counts))), color="salmon", edgecolor="white")
axes[1].set_xlabel("frames");       axes[1].set_title("Frames per episode")
short = [l[:28]+"..." if len(l)>28 else l for l in task_counts]
axes[2].bar(range(len(task_counts)), list(task_counts.values()), color="mediumpurple", edgecolor="white")
axes[2].set_xticks(range(len(task_counts)))
axes[2].set_xticklabels(short, rotation=20, ha="right", fontsize=8)
axes[2].set_title("Episodes per task")
plt.tight_layout(); plt.show()

print(f"Mean duration : {np.mean(durations):.2f}s")
print(f"Mean frames   : {np.mean(frame_counts):.0f}")